# S&P 500 Returns and U.S. Macroeconomic Indicators — Stationary Tests

**Objective:** Apply the  Jarque-Bera test to assess normality, and perform ADF and KPSS tests to evaluate unit roots in the time series.

## 1. Load Cleaned Dataset

In [1]:
%pip install statsmodels scipy matplotlib
import pandas as pd

df = pd.read_csv("sp500_macro_monthly_1990_2026.csv", index_col=0, parse_dates = True)
df.head()

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.1.1 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


,unemployment_rate,fed_rate,indpro,gs10,usrec,sp500_close,inflation_yoy,sp500_log_return,log_indpro
1991-01-01,6.4,6.91,61.1355,8.09,1.0,343.929993,5.647059,4.067902,4.113093
1991-02-01,6.6,6.25,60.6838,7.85,1.0,367.070007,5.312500,6.511446,4.105677
1991-03-01,6.8,6.12,60.3346,8.11,1.0,375.220001,4.821151,2.195994,4.099906
1991-04-01,6.7,5.91,60.4938,8.04,0.0,375.339996,4.809930,0.031975,4.102541
1991-05-01,6.9,5.78,61.0633,8.07,0.0,389.829987,5.034857,3.787844,4.111911


## 2. Normality Test (Jarque-Bera)

Formal test confirming the visual patterns observed in the histograms and Q-Q plots from the exploratory analysis notebook. H0: the series is normally distributed.
Also, the variable 'usrec' is excluded because this one is a dummy, not a continuos variable subject to a normal distribution

In [2]:
from scipy.stats import jarque_bera

cols_to_test = ['sp500_log_return', 'unemployment_rate', 'fed_rate', 'indpro', 'log_indpro', 'gs10', 'inflation_yoy']

for col in cols_to_test:
    stat, p_value = jarque_bera(df[col].dropna())
    result = "Normal" if p_value > 0.05 else "Not Normal"
    print(f"{col:20s} -> stat={stat:8.2f}, p-value={p_value:.6f}  ({result})")


sp500_log_return     -> stat=   75.44, p-value=0.000000  (Not Normal)
unemployment_rate    -> stat=  188.07, p-value=0.000000  (Not Normal)
fed_rate             -> stat=   43.00, p-value=0.000000  (Not Normal)
indpro               -> stat=  105.72, p-value=0.000000  (Not Normal)
log_indpro           -> stat=  148.21, p-value=0.000000  (Not Normal)
gs10                 -> stat=   13.90, p-value=0.000960  (Not Normal)
inflation_yoy        -> stat=  310.66, p-value=0.000000  (Not Normal)


## 3. Stationary Test ADF and KPSS
ADF: H0 = the series has a unit root (non-stationary). We want to reject the null hypothesis (p<0.05).
KPSS: H0 = the series is stationary. We don't want to reject the null hypothesis (p > 0.05)
In this case we are going to use them together because they have opposite null hypothesis. Therefore, agreement between them increases confidence in the conclusion.

In [3]:
%pip install statsmodels scipy matplotlib

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.1.1 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [4]:
from statsmodels.tsa.stattools import adfuller, kpss

def test_stationarity(series, name):
    adf_stat, adf_p, *_ = adfuller(series.dropna())
    kpss_stat, kpss_p, *_ = kpss(series.dropna(), regression='c', nlags='auto')

    adf_result = "Stationarity" if adf_p < 0.05 else "NOT stationary"
    kpss_result = "Stationarity" if kpss_p > 0.05 else "NOT stationary"

    print(f"{name:20s} | ADF p={adf_p:.4f} ({adf_result:15s}) | KPSS p={kpss_p:.4f} ({kpss_result})")


In [5]:
for col in ['sp500_log_return', 'unemployment_rate', 'fed_rate', 'indpro', 'log_indpro', 'gs10', 'inflation_yoy']:
    test_stationarity(df[col],col)

sp500_log_return     | ADF p=0.0000 (Stationarity   ) | KPSS p=0.1000 (Stationarity)
unemployment_rate    | ADF p=0.0760 (NOT stationary ) | KPSS p=0.1000 (Stationarity)
fed_rate             | ADF p=0.0239 (Stationarity   ) | KPSS p=0.0100 (NOT stationary)
indpro               | ADF p=0.0962 (NOT stationary ) | KPSS p=0.0100 (NOT stationary)
log_indpro           | ADF p=0.0258 (Stationarity   ) | KPSS p=0.0100 (NOT stationary)
gs10                 | ADF p=0.1634 (NOT stationary ) | KPSS p=0.0100 (NOT stationary)
inflation_yoy        | ADF p=0.0051 (Stationarity   ) | KPSS p=0.1000 (Stationarity)


C:\Users\jmich\AppData\Local\Temp\ipykernel_492\3394606429.py:5: InterpolationWarning: The test statistic is outside of the range of p-values available in the
look-up table. The actual p-value is greater than the p-value returned.

  kpss_stat, kpss_p, *_ = kpss(series.dropna(), regression='c', nlags='auto')
C:\Users\jmich\AppData\Local\Temp\ipykernel_492\3394606429.py:5: InterpolationWarning: The test statistic is outside of the range of p-values available in the
look-up table. The actual p-value is greater than the p-value returned.

  kpss_stat, kpss_p, *_ = kpss(series.dropna(), regression='c', nlags='auto')
C:\Users\jmich\AppData\Local\Temp\ipykernel_492\3394606429.py:5: InterpolationWarning: The test statistic is outside of the range of p-values available in the
look-up table. The actual p-value is smaller than the p-value returned.

  kpss_stat, kpss_p, *_ = kpss(series.dropna(), regression='c', nlags='auto')
C:\Users\jmich\AppData\Local\Temp\ipykernel_492\3394606429.py:5: Inter

Based on ADF and KPSS stationary tests the series `sp500_log_return` and `inflation_yoy` are confirmed to be stationary. In contrast, `indpro` and `gs10` exhibit non-stationary dynamics. Finally, the results for `unemployment_rate`, `fed_rate`, and `log_indpro` remain inconclusive due to contradictions between the two tests.

## 4. First Differences for Non-Stationary Variables

Based on ADF/KPSS results, the following variables are treated as non-stationary (either both tests agree, or the tests conflict, which is treated conservatively as evidence of a unit root): `unemployment_rate`, `fed_rate`, `indpro`, `log_indpro`, `gs10`. First differences are computed and re-tested.

In [6]:
vars_to_diff = ['unemployment_rate', 'fed_rate', 'indpro', 'log_indpro', 'gs10']

for col in vars_to_diff:
    df[f"{col}_diff"] = df[col].diff()

df_test = df.dropna()

for col in vars_to_diff:
    test_stationarity(df_test[f"{col}_diff"], f'{col}_diff')

unemployment_rate_diff | ADF p=0.0000 (Stationarity   ) | KPSS p=0.1000 (Stationarity)
fed_rate_diff        | ADF p=0.0003 (Stationarity   ) | KPSS p=0.1000 (Stationarity)
indpro_diff          | ADF p=0.0000 (Stationarity   ) | KPSS p=0.1000 (Stationarity)
log_indpro_diff      | ADF p=0.0000 (Stationarity   ) | KPSS p=0.0453 (NOT stationary)
gs10_diff            | ADF p=0.0000 (Stationarity   ) | KPSS p=0.1000 (Stationarity)


C:\Users\jmich\AppData\Local\Temp\ipykernel_492\3394606429.py:5: InterpolationWarning: The test statistic is outside of the range of p-values available in the
look-up table. The actual p-value is greater than the p-value returned.

  kpss_stat, kpss_p, *_ = kpss(series.dropna(), regression='c', nlags='auto')
C:\Users\jmich\AppData\Local\Temp\ipykernel_492\3394606429.py:5: InterpolationWarning: The test statistic is outside of the range of p-values available in the
look-up table. The actual p-value is greater than the p-value returned.

  kpss_stat, kpss_p, *_ = kpss(series.dropna(), regression='c', nlags='auto')
C:\Users\jmich\AppData\Local\Temp\ipykernel_492\3394606429.py:5: InterpolationWarning: The test statistic is outside of the range of p-values available in the
look-up table. The actual p-value is greater than the p-value returned.

  kpss_stat, kpss_p, *_ = kpss(series.dropna(), regression='c', nlags='auto')
C:\Users\jmich\AppData\Local\Temp\ipykernel_492\3394606429.py:5: Inter

## 5. Final Decision on Variable Transformations

Based on the results above, `log_indpro` is discarded in favor of `indpro` (used as `indpro_diff`): the logarithmic transformation did not improve normality (Jarque-Bera: 148.21 vs. 105.72 for the level series) and `log_indpro_diff` remains non-stationary according to KPSS even after first-differencing, while `indpro_diff` is stationary under both ADF and KPSS.

**Final variable set for subsequent models:**
- `sp500_log_return` (level — already stationary)
- `inflation_yoy` (level — already stationary)
- `unemployment_rate_diff`, `fed_rate_diff`, `indpro_diff`, `gs10_diff` (first differences — non-stationary in levels)
- `usrec` (level — binary dummy, not subject to stationarity testing)